In [10]:
import pandas as pd
import re

# 1. Load the dataset
csv_filename = "long_beach_california_cats_with_tnr_all_years_2.csv"
try:
    df = pd.read_csv(csv_filename, low_memory=False)
except FileNotFoundError:
    df = pd.read_csv("long_beach_california_cats_with_tnr_all_years.csv", low_memory=False)

# Normalize column names
df.columns = [c.lower().strip() for c in df.columns]

# 2. Define target ZIP codes and household counts
household_data = {
    '90808': 14198,
    '90815': 15432,
    '90803': 16894,
    '90814': 9167,
    '90807': 12636,
    '90806': 12586,
    '90810': 9956,
    '90802': 22421,
    '90805': 28213,
    '90804': 15290,
    '90813': 17252
}
target_zips = list(household_data.keys())

# 3. Filter for Intake Type: Stray OR Trap and Return
valid_intakes = ['STRAY', 'TRAP, NEUTER, RETURN', 'TRAP, NEUTER, RELEASE']
df_filtered = df[df['intake_type'].astype(str).str.upper().isin(valid_intakes)].copy()

# 4. Filter for Alive Outcomes
df_filtered = df_filtered[
    (df_filtered['outcome_is_alive'] == True) | 
    (df_filtered['was_outcome_alive'] == 1)
].copy()

# 5. Extract 5-digit ZIP code from 'crossing' and keep only target ZIPs
df_filtered['zip_code'] = df_filtered['crossing'].astype(str).str.extract(r'(908\d{2})')[0]
df_filtered = df_filtered[df_filtered['zip_code'].isin(target_zips)].copy()

# 6. Parse dates and filter for 2017 and later (drops 2014, 2016, etc.)
df_filtered['intake_date'] = pd.to_datetime(df_filtered['intake_date'], errors='coerce')
df_filtered = df_filtered[df_filtered['intake_date'].dt.year >= 2017].copy()

# 7. Drop missing values from critical columns
required_columns = ['zip_code', 'intake_date', 'intake_type', 'outcome_type']
df_filtered = df_filtered.dropna(subset=required_columns).copy()

# Format as Year-Month (e.g., '2017-01')
df_filtered['year_month'] = df_filtered['intake_date'].dt.to_period('M').astype(str)

# -------------------------------------------------------------
# Output 1: Long-Format Monthly Summary Table
# -------------------------------------------------------------
monthly_summary = (
    df_filtered.groupby(['zip_code', 'year_month'])
    .size()
    .reset_index(name='cat_count')
)

monthly_summary['households'] = monthly_summary['zip_code'].map(household_data)
monthly_summary['estimated_feral_cats'] = (
    monthly_summary['households'] * 0.10 * 3.4
).round(2)

print("=== MONTHLY SUMMARY (2017 onwards) ===")
print(monthly_summary.head(15).to_string(index=False))

# -------------------------------------------------------------
# Output 2: Wide Matrix (ZIPs as rows, Months as columns)
# -------------------------------------------------------------
pivot_counts = df_filtered.pivot_table(
    index='zip_code', 
    columns='year_month', 
    values='animal_id', 
    aggfunc='count', 
    fill_value=0
)

# Attach household and estimated feral count columns at the beginning
pivot_counts.insert(0, 'households', pivot_counts.index.map(household_data))
pivot_counts.insert(1, 'estimated_feral_cats', (pivot_counts['households'] * 0.10 * 3.4).round(2))

print("\n=== PIVOT TABLE (ZIP x MONTH) ===")
display(pivot_counts)

=== MONTHLY SUMMARY (2017 onwards) ===
zip_code year_month  cat_count  households  estimated_feral_cats
   90802    2017-01          5       22421               7623.14
   90802    2017-02          9       22421               7623.14
   90802    2017-03          5       22421               7623.14
   90802    2017-04         15       22421               7623.14
   90802    2017-05          3       22421               7623.14
   90802    2017-06         14       22421               7623.14
   90802    2017-07         17       22421               7623.14
   90802    2017-08         10       22421               7623.14
   90802    2017-09         11       22421               7623.14
   90802    2017-10          5       22421               7623.14
   90802    2017-11          7       22421               7623.14
   90802    2017-12          5       22421               7623.14
   90802    2018-01          4       22421               7623.14
   90802    2018-02         13       22421         

year_month,households,estimated_feral_cats,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,...,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09,2026-10
zip_code,,,,,,,,,,,,,,,,,,,,,
90802,22421,7623.14,5,9,5,15,3,14,17,10,...,3,1,1,10,10,19,5,3,0,0
90803,16894,5743.96,3,1,2,0,4,2,9,12,...,0,1,0,2,1,0,1,1,2,0
90804,15290,5198.60,5,7,7,13,15,26,12,31,...,3,2,8,2,27,13,11,1,2,0
90805,28213,9592.42,31,30,29,49,100,79,67,32,...,8,13,31,38,45,58,28,45,26,2
90806,12586,4279.24,2,8,6,24,25,47,21,11,...,22,3,5,25,24,13,25,12,14,4
90807,12636,4296.24,4,0,13,6,10,8,8,14,...,1,2,10,2,11,8,0,0,1,1
90808,14198,4827.32,5,2,8,10,5,20,12,15,...,2,6,10,20,18,14,18,5,5,0
90810,9956,3385.04,1,2,16,28,25,21,10,19,...,5,7,4,36,19,26,4,2,3,3
90813,17252,5865.68,16,8,17,40,57,46,36,39,...,13,14,27,42,19,16,25,12,13,5


In [15]:
from pathlib import Path

output_path = Path("long_beach_FILTERED_ONLY.csv").resolve()

df_filtered.to_csv(output_path, index=False)

print(f"Original rows: {len(df):,}")
print(f"Filtered rows: {len(df_filtered):,}")
print(f"Saved to: {output_path}")

Original rows: 27,418
Filtered rows: 14,931
Saved to: /home/ibm-user/long_beach_FILTERED_ONLY.csv
